# Part A — Stage 1 with automated verification (Change Brief §4, §6)

The human sign-off gate is gone. Stage 1 now reads every page twice (layout +
structured facts), cross-checks the numbers, and writes `verification.json` +
`review.html` per document. This cell runs Stage 1 on one document and prints
the **verification summary table** (pages passed / flagged / reasons).

> Run from the service root (`services/rag-ingestion/`). Stage 1 calls Vertex, so
> it needs GCP creds + the source PDF in `data/00_raw/`. The summary table itself
> reads `verification.json` and runs offline once a document has been parsed.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DOC = 'mihp_i'  # pick any doc_id from config/documents.yaml

### Run Stage 1 (parse + auto-verify) on one document
Uncomment to actually parse (needs Vertex + the PDF). It prints a per-doc
`verify -> N pass, M flagged (reasons)` line at the end automatically.

In [ ]:
# from types import SimpleNamespace
# from pipeline import stage1_parse
# stage1_parse.run(SimpleNamespace(doc=DOC, version=None, force=False))

# Or, from a shell:  python cli.py stage1 --doc mihp_i

### Verification summary table
Reads each document's `verification.json`. Re-run verification without re-parsing
with `python cli.py verify --doc <id> --report`.

In [ ]:
parsed_root = ROOT / 'data' / '01_parsed'
reports = sorted(parsed_root.glob('*/verification.json')) if parsed_root.exists() else []

if not reports:
    print('No verification.json yet — run Stage 1 on a document first (cell above).')
else:
    hdr = f"{'doc_id':<28} {'pages':>5} {'pass':>5} {'flag':>5}  reasons"
    print(hdr); print('-' * len(hdr))
    for rp in reports:
        v = json.loads(rp.read_text())
        s = v['summary']
        reasons = ', '.join(s['reasons']) or '—'
        print(f"{v['doc_id']:<28} {s['pages']:>5} {s['passed']:>5} {s['flagged']:>5}  {reasons}")
        for pg in v['flagged_pages']:
            page = next(p for p in v['pages'] if p['page'] == pg)
            for c in page['checks']:
                if c['status'] == 'flag':
                    print(f"      page {pg}: [{c['category']}] {c['detail']}")